# HLLSet Algebra — Advanced Programming

This notebook explores the low-level Rust API of hllset-next:
TF vector, Commit chain, five-level rank algebra, per-register TF
ranking, and the Noether controller.

**Concepts:** TFVec (monotonic CRDT), Commit (D/R/N), rank algebra,
TfRegisterRanker, Noether controller (integer flux), Fisher matrix,
observable mask, IICA properties.

In [2]:
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-core" }
:dep hllset-ranks = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-ranks" }
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-dsl" }

use hllset_core::*;
use hllset_core::core::tfvec::TFVec;
use hllset_core::core::commit::Commit;
use hllset_ranks::*;
use hllset_ranks::token::*;
use hllset_ranks::bit::*;
use hllset_ranks::register::*;
use hllset_ranks::hllset::*;
use hllset_ranks::compound::*;
use hllset_ranks::fisher::*;
use hllset_ranks::mask::*;
use hllset_dsl::LatticeElement;

println!("All crates loaded — M={M}, TOTAL_BITS={}", hllset_core::core::hllset::TOTAL_BITS);

All crates loaded — M=1024, TOTAL_BITS=32768


---
## 1. TFVec — Bit-Level Term Frequency

The TF vector is the shared frequency table driving all rank computation.
Monotonic CRDT: values only increase, never decrease.
Wire format: 4-byte LE count + 32,768 × f64 = 262,148 bytes.

In [3]:
let mut tf = TFVec::new();
println!("Entries: {}", tf.len());
println!("Total: {:.1}", tf.total());

Entries: 32768
Total: 0.0


### Monotonic increment & CRDT merge

In [4]:
tf.increment(0, 1.0);
tf.increment(100, 5.0);
tf.increment(0, 2.0);
println!("TF[0]: {:.1}, TF[100]: {:.1}, Total: {:.1}", tf.get(0), tf.get(100), tf.total());

// CRDT merge: element-wise maximum
let mut tf_a = TFVec::new();
tf_a.increment(0, 10.0);
tf_a.increment(1, 5.0);
let mut tf_b = TFVec::new();
tf_b.increment(0, 7.0);
tf_b.increment(1, 12.0);
tf_a.merge(&tf_b);
assert!((tf_a.get(0) - 10.0).abs() < 1e-10);
assert!((tf_a.get(1) - 12.0).abs() < 1e-10);
println!("Merge: TF[0]={:.1}, TF[1]={:.1} (takes max)", tf_a.get(0), tf_a.get(1));

TF[0]: 3.0, TF[100]: 5.0, Total: 8.0
Merge: TF[0]=10.0, TF[1]=12.0 (takes max)


### Wire format roundtrip

In [5]:
let bytes = tf.to_bytes();
println!("Serialized: {} bytes", bytes.len());
let tf2 = TFVec::from_bytes(&bytes).unwrap();
assert_eq!(tf.values, tf2.values);
println!("Roundtrip OK — {} entries", tf2.len());

Serialized: 262148 bytes
Roundtrip OK — 32768 entries


---
## 2. Commit — D/R/N Lattice Evolution

Each evolution step produces a Commit recording D/R/N decomposition.
Commits form a content-addressed chain (`t:<sha1>`).

In [6]:
let prev = LatticeElement::from_tokens(&["a", "b", "c", "d"]);
let curr = LatticeElement::from_tokens(&["b", "c", "d", "e"]);

let r = prev.intersection(&curr);
let d = prev.difference(&curr);
let n = curr.difference(&prev);
println!("R: {} bits, D: {} bits, N: {} bits", r.popcount(), d.popcount(), n.popcount());

let commit = Commit::new(curr.key(), "t:prev_head", d.key(), r.key(), n.key());
println!("Commit key: {}", commit.content_key());
println!("Chain valid: {}", commit.chain_valid("t:prev_head"));
println!("JSON: {}", commit.to_json());

R: 3 bits, D: 1 bits, N: 1 bits
Commit key: t:95e977f07c53a4fae214f227d20e1ec3387c3493
Chain valid: true
JSON: {"ts":1785522169683189,"s":"h:bdd4dacb114808ca9eced48bb51b8cee1591b120","h":"t:prev_head","d":"h:5b2b1eab8e1a07fa2bb1274d7f7b5c3d0959367e","r":"h:ee28b7d0ef35ef4eb6b176ae644cf0e3a9e14746","n":"h:13c28a958fbf491ce5f43c359d042281b4927bcb"}


---
## 3. Five-Level Rank Algebra

| Level | Function | What it ranks |
|-------|----------|---------------|
| 1 | F(TF) | Token frequency |
| 2 | G({token-R}) | Bit position |
| 3 | H({bit-R}) | Register (32 bits) |
| 4 | K(degree) | HLLSet |
| 5 | L(max)/M(min) | Compound |

See `09_rank_algebra.ipynb` for full treatment. Below: the new TF→Register bridge.

---
## 4. Per-Register TF Ranking (TfRegisterRanker)

Bridges the 32,768-entry TF vector to 1,024 register-level ranks.
Enables register queries without a TokenLUT.

In [7]:
let mut tf = TFVec::new();
for pos in 0..8 { tf.increment(pos, 100.0); }     // reg 0: heavy
for pos in 32..40 { tf.increment(pos, 50.0); }    // reg 1: medium
for pos in 64..68 { tf.increment(pos, 10.0); }    // reg 2: light

let ranker = TfRegisterRanker::default();
let reg0 = ranker.rank_register(&tf, 0);
let reg1 = ranker.rank_register(&tf, 1);
let reg2 = ranker.rank_register(&tf, 2);
println!("Reg 0: rank={}, slots={}", reg0.value, reg0.active_slots);
println!("Reg 1: rank={}, slots={}", reg1.value, reg1.active_slots);
println!("Reg 2: rank={}, slots={}", reg2.value, reg2.active_slots);
assert!(reg0.value > reg1.value && reg1.value > reg2.value);
println!("Ranking preserves TF ordering");

let top5 = ranker.top_k(&tf, 5);
println!("Top-5: {:?}", top5.iter().map(|r| (r.register, r.value)).collect::<Vec<_>>());

Reg 0: rank=800, slots=8
Reg 1: rank=400, slots=8
Reg 2: rank=40, slots=4
Ranking preserves TF ordering
Top-5: [(0, 800), (1, 400), (2, 40), (3, 0), (4, 0)]


---
## 5. Noether Controller (Integer Flux)

Monitors key creation/eviction rate with integer-only arithmetic.
Integer halving decay replaces float 0.9 exponential decay.

In [8]:
struct FluxMonitor { flux: i64, threshold: i64 }
impl FluxMonitor {
    fn new(t: i64) -> Self { Self { flux: 0, threshold: t } }
    fn record_new(&mut self) { self.flux += 1; }
    fn record_evict(&mut self) { self.flux -= 1; }
    fn tick(&mut self) -> bool {
        let drift = self.flux.abs() > self.threshold;
        self.flux /= 2;
        drift
    }
}

let mut m = FluxMonitor::new(5);
for _ in 0..10 { m.record_new(); }
println!("After 10 inserts: flux={}", m.flux);

let drift1 = m.tick();
println!("Tick 1: flux={}, drift={}", m.flux, drift1);
let drift2 = m.tick();
println!("Tick 2: flux={}, drift={}", m.flux, drift2);

for _ in 0..5 { m.tick(); }
println!("After 5 more ticks: flux={}", m.flux);
assert_eq!(m.flux, 0);
println!("Integer flux decays to zero — no floating point");

After 10 inserts: flux=10
Tick 1: flux=5, drift=true
Tick 2: flux=2, drift=false
After 5 more ticks: flux=0
Integer flux decays to zero — no floating point


---
## 6. Fisher Matrix — Cross-Layer Bit Coupling

Counts how many layers have both bits b and b' set simultaneously.
High diagonal = persistent bits. High off-diagonal = co-occurring bits.

In [9]:
let mut fisher = FisherMatrix::new();

// Add 3 layers (observations at different times)
let l0 = LatticeElement::from_tokens(&["urgent", "now", "alert"]);
let l1 = LatticeElement::from_tokens(&["urgent", "later", "review"]);
let l2 = LatticeElement::from_tokens(&["archive", "later", "done"]);

fisher.add_layer(&l0);
fisher.add_layer(&l1);
fisher.add_layer(&l2);

println!("Layers: {}", fisher.layer_count());
println!("Non-zero co-occurrence entries: {}", fisher.entry_count());

// Most persistent bits (appear in most layers)
let persistent = fisher.most_persistent(3);
println!("Most persistent bit positions: {:?}", persistent);

// Check persistence of a specific position from l0
if let Some(first_pos) = l0.hllset().active_positions().first() {
    let diag = fisher.diagonal(*first_pos);
    println!("Persistence of first bit ({:?}): {} layers", first_pos, diag);
}

Layers: 3
Non-zero co-occurrence entries: 16
Most persistent bit positions: [((967, 1), 2), ((352, 3), 2), ((892, 0), 1)]
Persistence of first bit ((352, 3)): 2 layers


()

---
## 7. Observable Mask — Rank Depletion

The set of HLLSets with rank above threshold θ. Controls attention,
not existence — every HLLSet remains retrievable by its content key.

In [10]:
let mut idx = HLLSetRankIndex::new();
for (key, value, degree) in [
    ("h:hot", 100u64, 3usize),
    ("h:warm", 80, 4),
    ("h:cool", 60, 2),
    ("h:cold", 40, 1),
    ("h:frozen", 20, 1),
] {
    let r = HLLSetRank::from_raw(key, degree, value, &DegreeRankFn);
    idx.insert(r);
}

let mask50 = ObservableMask::apply(&idx, 50);
let mask90 = ObservableMask::apply(&idx, 90);
println!("At θ=50: {}/{} observable", mask50.observable_count(), mask50.total);
println!("At θ=90: {}/{} observable", mask90.observable_count(), mask90.total);
assert!(mask50.observable_count() > mask90.observable_count());
println!("Lower threshold = more observable");

At θ=50: 0/5 observable
At θ=90: 0/5 observable



thread '<unnamed>' (64224) panicked at src/lib.rs:183:1:
assertion failed: mask50.observable_count() > mask90.observable_count()
stack backtrace:
   0: __rustc::rust_begin_unwind
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/std/src/panicking.rs:689:5
   1: core::panicking::panic_fmt
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/core/src/panicking.rs:80:14
   2: core::panicking::panic
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/core/src/panicking.rs:150:5
   3: run_user_code_9
   4: evcxr::runtime::Runtime::run_loop
   5: evcxr::runtime::runtime_hook
   6: evcxr_jupyter::main
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


---
## 8. IICA Properties

Every operation satisfies Idempotency, Immutability, Content-Addressability.
These compose: if each step is IICA, the entire pipeline is IICA.

In [11]:
let h1 = HLLSet::from_tokens(&["alice", "bob", "carol"]);
let h2 = HLLSet::from_tokens(&["alice", "bob", "carol"]);
assert_eq!(h1.content_key(), h2.content_key());
assert_eq!(h1.union(&h1).popcount(), h1.popcount());
println!("IICA verified");
println!("Key: {}", h1.content_key());
println!("popcount(A u A) = popcount(A) = {}", h1.popcount());

IICA verified
Key: h:000a705032649d985683fcc9ae00732b5a7e7906
popcount(A u A) = popcount(A) = 3


---
## Summary

| Component | Key property |
|-----------|-------------|
| **TFVec** | Monotonic CRDT, 262KB wire format |
| **Commit** | D/R/N content-addressed chain |
| **Rank algebra** | Integer-only, FPGA-native |
| **TfRegisterRanker** | TF → register rank, no LUT |
| **Noether** | Integer flux, halving decay |
| **Fisher** | Cross-layer bit co-occurrence |
| **Observable mask** | Rank-threshold attention filter |

All built on IICA: idempotent, immutable, content-addressed.